In [1]:
"""
Local Curvature Analysis of Cosmic Ray Time Series
====================================================
Production code for real NMDB + OMNI data.

Pipeline:
  1. Load data (JUNG, MOSC + OMNI variables)
  2. Segment by solar cycle
  3. Build Visibility Graphs per station per cycle
  4. Compute local curvatures (Forman-Ricci, Ollivier-Ricci, Ricci Flow)
  5. Generate curvature time series
  6. Correlate with space weather variables
  7. Anomaly detection (Forbush decrease candidates)

Expected input DataFrame format:
  Index: DATETIME (daily)
  Columns: JUNG, MOSC, Scalar B (nT), SW Plasma Temperature (K),
           SW Proton Density (N/cm^3), SW Plasma Speed (km/s),
           R (Sunspot No.), Dst-index (nT), Cycle

Author: D. Sierra-Porta
"""

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.optimize import linprog
from scipy.stats import pearsonr, spearmanr, kendalltau
from scipy.signal import savgol_filter
from collections import defaultdict
import warnings
import time
import os

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.figsize': (14, 5),
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'legend.fontsize': 10,
    'figure.dpi': 150,
    'savefig.dpi': 150,
    'savefig.bbox': 'tight'
})


# ============================================================
# SECTION 1: DATA LOADING AND PREPARATION
# ============================================================

def load_and_prepare_data(filepath):
    """
    Load the merged NMDB + OMNI dataset.
    
    Parameters
    ----------
    filepath : str
        Path to CSV/pickle file with the merged data.
        Expected columns: JUNG, MOSC, Scalar B (nT), 
        SW Plasma Temperature (K), SW Proton Density (N/cm^3),
        SW Plasma Speed (km/s), R (Sunspot No.), Dst-index (nT), Cycle
        Index: DATETIME (daily)
    
    Returns
    -------
    df : pd.DataFrame
        Cleaned daily data
    """
    # Adjust read function based on file extension
    ext = os.path.splitext(filepath)[1].lower()
    if ext == '.csv':
        df = pd.read_csv(filepath, index_col='DATETIME', parse_dates=True)
    elif ext in ['.pkl', '.pickle']:
        df = pd.read_pickle(filepath)
    elif ext == '.parquet':
        df = pd.read_parquet(filepath)
    else:
        # Try CSV as default
        df = pd.read_csv(filepath, index_col=0, parse_dates=True)
    
    # Standardize column names for internal use
    col_map = {
        'Scalar B, nT': 'B',
        'SW Plasma Temperature, K': 'Temp',
        'SW Proton Density, N/cm^3': 'Density',
        'SW Plasma Speed, km/s': 'Speed',
        'R (Sunspot No.)': 'SSN',
        'Dst-index, nT': 'Dst'
    }
    df = df.rename(columns=col_map)
    
    print(f"Data loaded: {df.shape[0]} rows, {df.shape[1]} columns")
    print(f"Date range: {df.index.min()} to {df.index.max()}")
    print(f"Cycles present: {sorted(df['Cycle'].dropna().unique().astype(int))}")
    print(f"\nMissing data (%):")
    for col in df.columns:
        pct = df[col].isna().sum() / len(df) * 100
        print(f"  {col:30s}: {pct:.1f}%")
    
    return df


def resample_data(df, freq='W'):
    """
    Resample to desired frequency.
    
    Parameters
    ----------
    freq : str
        'W' for weekly, 'M' for monthly, 'D' for daily (no resampling)
    """
    if freq == 'D':
        return df.copy()
    
    # For Cycle column, use the mode (most frequent value)
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if 'Cycle' in numeric_cols:
        numeric_cols.remove('Cycle')
    
    df_resampled = df[numeric_cols].resample(freq).mean()
    
    # Add Cycle back
    if 'Cycle' in df.columns:
        df_resampled['Cycle'] = df['Cycle'].resample(freq).apply(
            lambda x: x.mode()[0] if len(x.mode()) > 0 else np.nan
        )
    
    return df_resampled


def get_cycle_data(df, cycle, cr_col='JUNG'):
    """
    Extract clean CR series for a specific solar cycle.
    
    Returns
    -------
    series : pd.Series
        CR counts for the cycle (NaNs removed via interpolation)
    dates : pd.DatetimeIndex
    """
    mask = df['Cycle'] == cycle
    cycle_data = df.loc[mask].copy()
    
    if cycle_data.empty:
        print(f"  WARNING: No data for cycle {cycle}")
        return None, None
    
    # Interpolate small gaps (< 5 consecutive NaNs)
    series = cycle_data[cr_col].interpolate(method='linear', limit=5)
    
    # Drop remaining NaNs from edges
    series = series.dropna()
    
    if len(series) < 20:
        print(f"  WARNING: Cycle {cycle}, {cr_col} has only {len(series)} valid points")
        return None, None
    
    return series, series.index


# ============================================================
# SECTION 2: CURVATURE IMPLEMENTATIONS
# ============================================================

def forman_ricci_local(G):
    """
    Compute Forman-Ricci curvature for each edge in an unweighted graph.
    
    For unweighted graphs:
        F(e=(u,v)) = |common_neighbors(u,v)| + 2 - |symmetric_diff_neighbors(u,v)|
    
    Then aggregate to nodes by averaging adjacent edge curvatures.
    
    Parameters
    ----------
    G : nx.Graph
        Unweighted graph (from visibility_graph)
    
    Returns
    -------
    edge_curvatures : dict
        {(u, v): curvature_value}
    node_curvatures : dict
        {node: mean_curvature_of_adjacent_edges}
    """
    edge_curvatures = {}
    
    for u, v in G.edges():
        neighbors_u = set(G.neighbors(u)) - {v}
        neighbors_v = set(G.neighbors(v)) - {u}
        
        common = len(neighbors_u & neighbors_v)
        non_common = len(neighbors_u.symmetric_difference(neighbors_v))
        
        edge_curvatures[(u, v)] = common + 2 - non_common
    
    # Aggregate to nodes: average curvature of all adjacent edges
    node_curvatures = {}
    for n in G.nodes():
        adj_curvs = []
        for u, v in G.edges(n):
            key = (u, v) if (u, v) in edge_curvatures else (v, u)
            if key in edge_curvatures:
                adj_curvs.append(edge_curvatures[key])
        
        node_curvatures[n] = np.mean(adj_curvs) if adj_curvs else 0.0
    
    return edge_curvatures, node_curvatures


def _wasserstein_1d(mu, nu, cost_matrix):
    """
    Compute 1st Wasserstein distance between two discrete distributions
    via linear programming (scipy.optimize.linprog).
    """
    n = len(mu)
    m = len(nu)
    c = cost_matrix.flatten()
    
    A_eq = np.zeros((n + m, n * m))
    for i in range(n):
        A_eq[i, i * m:(i + 1) * m] = 1
    for j in range(m):
        A_eq[n + j, j::m] = 1
    
    b_eq = np.concatenate([mu, nu])
    
    result = linprog(c, A_eq=A_eq, b_eq=b_eq,
                     bounds=[(0, None)] * (n * m),
                     method='highs', options={'presolve': True})
    
    return result.fun if result.success else np.nan


def ollivier_ricci_local(G, alpha=0.5, max_edges=600, verbose=True):
    """
    Compute Ollivier-Ricci curvature for each edge using optimal transport.
    
    Parameters
    ----------
    G : nx.Graph
    alpha : float
        Laziness parameter in [0, 1]. Default 0.5.
    max_edges : int
        If graph has more edges, sample this many.
    verbose : bool
    
    Returns
    -------
    edge_curvatures : dict
    node_curvatures : dict
    """
    nodes = list(G.nodes())
    n = len(nodes)
    node_idx = {node: i for i, node in enumerate(nodes)}
    
    # All-pairs shortest paths (BFS for unweighted)
    if verbose:
        print(f"  [OR] Computing shortest paths ({n} nodes)...")
    dist_dict = dict(nx.all_pairs_shortest_path_length(G))
    
    # Build distance matrix
    dist_matrix = np.full((n, n), np.inf)
    for u in nodes:
        for v, d in dist_dict[u].items():
            dist_matrix[node_idx[u]][node_idx[v]] = d
    
    edges = list(G.edges())
    if len(edges) > max_edges:
        if verbose:
            print(f"  [OR] Sampling {max_edges}/{len(edges)} edges...")
        sample_idx = np.random.choice(len(edges), max_edges, replace=False)
        edges_to_compute = [edges[i] for i in sample_idx]
    else:
        edges_to_compute = edges
    
    if verbose:
        print(f"  [OR] Computing curvature for {len(edges_to_compute)} edges...")
    
    edge_curvatures = {}
    
    for idx, (u, v) in enumerate(edges_to_compute):
        if verbose and idx % 500 == 0 and idx > 0:
            print(f"    ... {idx}/{len(edges_to_compute)}")
        
        # Probability measure at u
        neighbors_u = list(G.neighbors(u))
        support_u = [u] + neighbors_u
        mu = np.zeros(len(support_u))
        mu[0] = alpha
        if neighbors_u:
            mu[1:] = (1 - alpha) / len(neighbors_u)
        else:
            mu[0] = 1.0
        
        # Probability measure at v
        neighbors_v = list(G.neighbors(v))
        support_v = [v] + neighbors_v
        nu = np.zeros(len(support_v))
        nu[0] = alpha
        if neighbors_v:
            nu[1:] = (1 - alpha) / len(neighbors_v)
        else:
            nu[0] = 1.0
        
        # Cost matrix
        cost = np.zeros((len(support_u), len(support_v)))
        for i, su in enumerate(support_u):
            for j, sv in enumerate(support_v):
                cost[i, j] = dist_matrix[node_idx[su]][node_idx[sv]]
        
        W = _wasserstein_1d(mu, nu, cost)
        d_uv = dist_matrix[node_idx[u]][node_idx[v]]
        
        if d_uv > 0 and not np.isnan(W):
            edge_curvatures[(u, v)] = 1.0 - W / d_uv
        else:
            edge_curvatures[(u, v)] = 0.0
    
    # Aggregate to nodes
    node_curvatures = {}
    for nd in G.nodes():
        adj_curv = []
        for (u, v), c in edge_curvatures.items():
            if u == nd or v == nd:
                adj_curv.append(c)
        node_curvatures[nd] = np.mean(adj_curv) if adj_curv else np.nan
    
    return edge_curvatures, node_curvatures


def ricci_flow_local(G, alpha=0.5, iterations=20, verbose=True):
    """
    Compute discrete Ricci flow: iteratively adjust edge weights
    to uniformize Ollivier-Ricci curvature.
    
    Parameters
    ----------
    G : nx.Graph
    alpha : float
    iterations : int
        Number of flow steps (paper uses 50, but 15-20 is often sufficient)
    
    Returns
    -------
    edge_flow : dict
        Final edge weights after flow
    node_flow : dict
        Node-level mean of adjacent final edge weights
    """
    G_flow = G.copy()
    
    for u, v in G_flow.edges():
        G_flow[u][v]['weight'] = 1.0
    
    if verbose:
        print(f"  [RF] Running Ricci flow ({iterations} iterations, {G.number_of_nodes()} nodes)...")
    
    nodes = list(G_flow.nodes())
    node_idx = {node: i for i, node in enumerate(nodes)}
    
    for it in range(iterations):
        if verbose and it % 5 == 0:
            print(f"    ... iteration {it}/{iterations}")
        
        # Weighted shortest paths
        dist_dict = dict(nx.all_pairs_dijkstra_path_length(G_flow, weight='weight'))
        
        for u, v in G_flow.edges():
            neighbors_u = list(G_flow.neighbors(u))
            neighbors_v = list(G_flow.neighbors(v))
            
            support_u = [u] + neighbors_u
            mu = np.zeros(len(support_u))
            mu[0] = alpha
            if neighbors_u:
                mu[1:] = (1 - alpha) / len(neighbors_u)
            else:
                mu[0] = 1.0
            
            support_v = [v] + neighbors_v
            nu = np.zeros(len(support_v))
            nu[0] = alpha
            if neighbors_v:
                nu[1:] = (1 - alpha) / len(neighbors_v)
            else:
                nu[0] = 1.0
            
            cost = np.zeros((len(support_u), len(support_v)))
            for i, su in enumerate(support_u):
                for j, sv in enumerate(support_v):
                    cost[i, j] = dist_dict.get(su, {}).get(sv, 1e6)
            
            W = _wasserstein_1d(mu, nu, cost)
            d_uv = dist_dict.get(u, {}).get(v, 1.0)
            
            kappa = (1.0 - W / d_uv) if (d_uv > 0 and not np.isnan(W)) else 0.0
            
            G_flow[u][v]['weight'] = max((1 - kappa) * d_uv, 1e-6)
    
    # Extract final weights
    edge_flow = {(u, v): G_flow[u][v]['weight'] for u, v in G_flow.edges()}
    
    # Node-level aggregation
    node_flow = {}
    for nd in G_flow.nodes():
        adj_w = [G_flow[u][v]['weight'] for u, v in G_flow.edges(nd)]
        node_flow[nd] = np.mean(adj_w) if adj_w else np.nan
    
    return edge_flow, node_flow


# ============================================================
# SECTION 3: CURVATURE TIME SERIES BUILDER
# ============================================================

def build_curvature_timeseries(series, dates, compute_or=True, compute_rf=False,
                                alpha=0.5, rf_iterations=20, rf_max_nodes=80,
                                or_max_edges=600, verbose=True):
    """
    Full pipeline: series -> VG -> curvatures -> curvature time series.
    
    Parameters
    ----------
    series : array-like
        CR count values
    dates : pd.DatetimeIndex
        Corresponding dates
    compute_or : bool
        Whether to compute Ollivier-Ricci (slower)
    compute_rf : bool
        Whether to compute Ricci Flow (slowest)
    alpha : float
        Laziness parameter for OR and RF
    rf_iterations : int
    rf_max_nodes : int
        If len(series) > rf_max_nodes, skip RF (too expensive)
    or_max_edges : int
        Max edges to sample for OR
    verbose : bool
    
    Returns
    -------
    result : dict with keys:
        'dates': DatetimeIndex
        'values': original series values
        'G': visibility graph
        'FR_node': Forman-Ricci node curvatures (array)
        'FR_edge': edge curvature dict
        'OR_node': Ollivier-Ricci node curvatures (array or None)
        'OR_edge': edge curvature dict or None
        'RF_node': Ricci Flow node values (array or None)
        'RF_edge': edge flow dict or None
        'n_nodes': number of nodes
        'n_edges': number of edges
    """
    values = np.array(series)
    n = len(values)
    
    if verbose:
        print(f"  Building VG ({n} nodes)...")
    
    t0 = time.time()
    G = nx.visibility_graph(values)
    t_vg = time.time() - t0
    
    if verbose:
        print(f"  VG: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges ({t_vg:.1f}s)")
    
    result = {
        'dates': dates,
        'values': values,
        'G': G,
        'n_nodes': G.number_of_nodes(),
        'n_edges': G.number_of_edges()
    }
    
    # --- Forman-Ricci (always computed, very fast) ---
    t0 = time.time()
    fr_edge, fr_node = forman_ricci_local(G)
    t_fr = time.time() - t0
    
    result['FR_edge'] = fr_edge
    result['FR_node'] = np.array([fr_node.get(i, np.nan) for i in range(n)])
    
    if verbose:
        print(f"  FR: mean={np.nanmean(result['FR_node']):.3f}, "
              f"std={np.nanstd(result['FR_node']):.3f} ({t_fr:.1f}s)")
    
    # --- Ollivier-Ricci ---
    if compute_or:
        t0 = time.time()
        or_edge, or_node = ollivier_ricci_local(G, alpha=alpha,
                                                  max_edges=or_max_edges,
                                                  verbose=verbose)
        t_or = time.time() - t0
        
        result['OR_edge'] = or_edge
        result['OR_node'] = np.array([or_node.get(i, np.nan) for i in range(n)])
        
        if verbose:
            print(f"  OR: mean={np.nanmean(result['OR_node']):.3f}, "
                  f"std={np.nanstd(result['OR_node']):.3f} ({t_or:.1f}s)")
    else:
        result['OR_edge'] = None
        result['OR_node'] = None
    
    # --- Ricci Flow ---
    if compute_rf and n <= rf_max_nodes:
        t0 = time.time()
        rf_edge, rf_node = ricci_flow_local(G, alpha=alpha,
                                             iterations=rf_iterations,
                                             verbose=verbose)
        t_rf = time.time() - t0
        
        result['RF_edge'] = rf_edge
        result['RF_node'] = np.array([rf_node.get(i, np.nan) for i in range(n)])
        
        if verbose:
            print(f"  RF: mean={np.nanmean(result['RF_node']):.3f}, "
                  f"std={np.nanstd(result['RF_node']):.3f} ({t_rf:.1f}s)")
    elif compute_rf and n > rf_max_nodes:
        if verbose:
            print(f"  RF: SKIPPED (n={n} > rf_max_nodes={rf_max_nodes})")
        result['RF_edge'] = None
        result['RF_node'] = None
    else:
        result['RF_edge'] = None
        result['RF_node'] = None
    
    return result


# ============================================================
# SECTION 4: CORRELATION ANALYSIS
# ============================================================

def compute_correlations(curv_series, omni_df, curv_dates, curv_label='FR'):
    """
    Correlate a curvature time series with OMNI space weather variables.
    
    Parameters
    ----------
    curv_series : np.array
        Curvature values (one per time step)
    omni_df : pd.DataFrame
        OMNI data with columns B, Temp, Density, Speed, SSN, Dst
    curv_dates : pd.DatetimeIndex
        Dates corresponding to curv_series
    curv_label : str
        Label for the curvature metric
    
    Returns
    -------
    results : pd.DataFrame
        Correlation results table
    """
    omni_vars = ['B', 'Temp', 'Density', 'Speed', 'SSN', 'Dst']
    omni_labels = ['Scalar B (nT)', 'SW Temperature (K)', 'Proton Density (N/cm³)',
                   'SW Speed (km/s)', 'Sunspot Number', 'Dst Index (nT)']
    
    records = []
    
    for var, label in zip(omni_vars, omni_labels):
        if var not in omni_df.columns:
            continue
        
        # Align by date
        omni_at_dates = omni_df[var].reindex(curv_dates)
        
        # Clean NaNs
        mask = ~(np.isnan(curv_series) | np.isnan(omni_at_dates.values))
        
        if mask.sum() < 10:
            continue
        
        x = omni_at_dates.values[mask]
        y = curv_series[mask]
        
        r_p, p_p = pearsonr(x, y)
        r_s, p_s = spearmanr(x, y)
        tau, p_tau = kendalltau(x, y)
        
        records.append({
            'Variable': label,
            'Curvature': curv_label,
            'N': int(mask.sum()),
            'Pearson_r': r_p,
            'Pearson_p': p_p,
            'Spearman_rho': r_s,
            'Spearman_p': p_s,
            'Kendall_tau': tau,
            'Kendall_p': p_tau
        })
    
    return pd.DataFrame(records)


def compute_cross_station_correlation(result_a, result_b, label_a='JUNG', label_b='MOSC'):
    """
    Correlate curvature time series between two stations.
    """
    records = []
    
    for metric in ['FR_node', 'OR_node', 'RF_node']:
        if result_a[metric] is None or result_b[metric] is None:
            continue
        
        # Align by overlapping dates
        dates_a = result_a['dates']
        dates_b = result_b['dates']
        common_dates = dates_a.intersection(dates_b)
        
        if len(common_dates) < 10:
            continue
        
        idx_a = [list(dates_a).index(d) for d in common_dates if d in dates_a]
        idx_b = [list(dates_b).index(d) for d in common_dates if d in dates_b]
        
        min_len = min(len(idx_a), len(idx_b))
        a = result_a[metric][idx_a[:min_len]]
        b = result_b[metric][idx_b[:min_len]]
        
        mask = ~(np.isnan(a) | np.isnan(b))
        if mask.sum() < 10:
            continue
        
        r_p, p_p = pearsonr(a[mask], b[mask])
        r_s, p_s = spearmanr(a[mask], b[mask])
        
        metric_name = metric.replace('_node', '')
        records.append({
            'Metric': metric_name,
            'Stations': f'{label_a} vs {label_b}',
            'N': int(mask.sum()),
            'Pearson_r': r_p,
            'Pearson_p': p_p,
            'Spearman_rho': r_s,
            'Spearman_p': p_s
        })
    
    return pd.DataFrame(records)


# ============================================================
# SECTION 5: ANOMALY DETECTION
# ============================================================

def detect_curvature_anomalies(curv_series, dates, sigma=2.0, metric_name='FR'):
    """
    Detect anomalous curvature values (potential Forbush decrease indicators).
    
    Parameters
    ----------
    curv_series : np.array
    dates : pd.DatetimeIndex
    sigma : float
        Number of standard deviations for threshold
    
    Returns
    -------
    anomalies : pd.DataFrame
        Dates and curvature values of detected anomalies
    threshold_low : float
    threshold_high : float
    """
    mean = np.nanmean(curv_series)
    std = np.nanstd(curv_series)
    
    threshold_low = mean - sigma * std
    threshold_high = mean + sigma * std
    
    # Low anomalies (potential FD events)
    mask_low = curv_series < threshold_low
    # High anomalies (potential GLE or recovery events)
    mask_high = curv_series > threshold_high
    
    anomalies_low = pd.DataFrame({
        'date': dates[mask_low],
        'curvature': curv_series[mask_low],
        'type': 'low',
        'metric': metric_name
    })
    
    anomalies_high = pd.DataFrame({
        'date': dates[mask_high],
        'curvature': curv_series[mask_high],
        'type': 'high',
        'metric': metric_name
    })
    
    anomalies = pd.concat([anomalies_low, anomalies_high]).sort_values('date')
    
    return anomalies, threshold_low, threshold_high


# ============================================================
# SECTION 6: VISUALIZATION
# ============================================================

def plot_curvature_timeseries(results_dict, station_labels, save_prefix='fig'):
    """
    Plot CR counts + local curvature time series for multiple stations.
    
    Parameters
    ----------
    results_dict : dict
        {station_label: result_dict from build_curvature_timeseries}
    station_labels : list of str
    save_prefix : str
    """
    n_stations = len(station_labels)
    n_panels = n_stations + 2  # CR panels + FR panel + OR panel
    
    fig, axes = plt.subplots(n_panels, 1, figsize=(16, 3.5 * n_panels), sharex=True)
    fig.suptitle('Local Curvature Dynamics: Cosmic Ray Time Series',
                 fontsize=14, fontweight='bold', y=0.98)
    
    colors = ['#1f77b4', '#2ca02c', '#d62728', '#9467bd', '#8c564b']
    
    # CR count panels
    for i, label in enumerate(station_labels):
        res = results_dict[label]
        axes[i].plot(res['dates'], res['values'], color=colors[i % len(colors)],
                     linewidth=0.8, alpha=0.8)
        axes[i].set_ylabel(f'CR Counts\n({label})')
        axes[i].set_title(f'({chr(97+i)}) {label} Cosmic Ray Intensity', loc='left', fontsize=11)
        axes[i].grid(True, alpha=0.3)
    
    # Forman-Ricci panel
    ax_fr = axes[n_stations]
    for i, label in enumerate(station_labels):
        res = results_dict[label]
        ax_fr.plot(res['dates'], res['FR_node'], color=colors[i % len(colors)],
                   linewidth=0.7, alpha=0.7, label=f'{label} (FR)')
    ax_fr.axhline(0, color='gray', linestyle=':', linewidth=0.5)
    ax_fr.set_ylabel('Forman-Ricci\nCurvature')
    ax_fr.set_title(f'({chr(97+n_stations)}) Local Forman-Ricci Curvature', loc='left', fontsize=11)
    ax_fr.legend(loc='upper right', fontsize=9)
    ax_fr.grid(True, alpha=0.3)
    
    # Ollivier-Ricci panel
    ax_or = axes[n_stations + 1]
    has_or = False
    for i, label in enumerate(station_labels):
        res = results_dict[label]
        if res['OR_node'] is not None:
            ax_or.plot(res['dates'], res['OR_node'], color=colors[i % len(colors)],
                       linewidth=0.7, alpha=0.7, label=f'{label} (OR)')
            has_or = True
    
    if has_or:
        ax_or.axhline(0, color='gray', linestyle=':', linewidth=0.5)
        ax_or.set_ylabel('Ollivier-Ricci\nCurvature')
        ax_or.set_title(f'({chr(97+n_stations+1)}) Local Ollivier-Ricci Curvature', loc='left', fontsize=11)
        ax_or.legend(loc='upper right', fontsize=9)
    ax_or.grid(True, alpha=0.3)
    ax_or.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.xticks(rotation=45)
    
    plt.tight_layout()
    plt.savefig(f'{save_prefix}_curvature_timeseries.png')
    plt.close()
    print(f"  Saved: {save_prefix}_curvature_timeseries.png")


def plot_curvature_vs_omni(result, omni_df, station_label, curv_metric='FR_node',
                            save_prefix='fig'):
    """
    Scatter plots of curvature vs each OMNI variable.
    """
    omni_vars = ['B', 'Speed', 'Dst', 'SSN', 'Density', 'Temp']
    omni_labels = ['Scalar B (nT)', 'SW Speed (km/s)', 'Dst (nT)',
                   'Sunspot No.', 'Proton Density', 'SW Temp (K)']
    
    available = [(v, l) for v, l in zip(omni_vars, omni_labels) if v in omni_df.columns]
    
    n_plots = len(available)
    ncols = 3
    nrows = (n_plots + ncols - 1) // ncols
    
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.5 * ncols, 4.5 * nrows))
    axes = axes.flatten() if n_plots > 1 else [axes]
    
    metric_label = curv_metric.replace('_node', '')
    fig.suptitle(f'{metric_label} Curvature ({station_label}) vs Space Weather Variables',
                 fontsize=13, fontweight='bold')
    
    curv = result[curv_metric]
    dates = result['dates']
    
    for idx, (var, label) in enumerate(available):
        ax = axes[idx]
        
        omni_at_dates = omni_df[var].reindex(dates)
        mask = ~(np.isnan(curv) | np.isnan(omni_at_dates.values))
        
        if mask.sum() < 5:
            ax.set_visible(False)
            continue
        
        x = omni_at_dates.values[mask]
        y = curv[mask]
        
        ax.scatter(x, y, s=10, alpha=0.4, c='steelblue', edgecolors='none')
        
        r_p, p_p = pearsonr(x, y)
        r_s, p_s = spearmanr(x, y)
        
        # Trend line
        z = np.polyfit(x, y, 1)
        p_fit = np.poly1d(z)
        x_sorted = np.sort(x)
        ax.plot(x_sorted, p_fit(x_sorted), 'r-', linewidth=1.5, alpha=0.7)
        
        sig_p = '***' if p_p < 0.001 else ('**' if p_p < 0.01 else ('*' if p_p < 0.05 else 'ns'))
        ax.set_title(f'{label}\nr={r_p:.3f} {sig_p}, ρ={r_s:.3f}', fontsize=10)
        ax.set_xlabel(label, fontsize=9)
        ax.set_ylabel(f'{metric_label} Curvature', fontsize=9)
        ax.grid(True, alpha=0.2)
    
    # Hide unused subplots
    for idx in range(len(available), len(axes)):
        axes[idx].set_visible(False)
    
    plt.tight_layout()
    plt.savefig(f'{save_prefix}_curvature_vs_omni_{station_label}.png')
    plt.close()
    print(f"  Saved: {save_prefix}_curvature_vs_omni_{station_label}.png")


def plot_anomaly_detection(result, omni_df, station_label, sigma=2.0, save_prefix='fig'):
    """
    Plot curvature anomalies aligned with Dst for Forbush decrease detection.
    """
    fig, axes = plt.subplots(3, 1, figsize=(16, 11), sharex=True)
    fig.suptitle(f'Curvature Anomaly Detection ({station_label})',
                 fontsize=13, fontweight='bold', y=0.96)
    
    dates = result['dates']
    
    # Panel a: CR counts
    axes[0].plot(dates, result['values'], 'b-', linewidth=0.8)
    axes[0].set_ylabel('CR Counts')
    axes[0].set_title(f'(a) {station_label} Cosmic Ray Intensity', loc='left')
    axes[0].grid(True, alpha=0.3)
    
    # Panel b: FR curvature with anomalies
    fr = result['FR_node']
    anomalies, thr_low, thr_high = detect_curvature_anomalies(fr, dates, sigma=sigma, metric_name='FR')
    
    axes[1].plot(dates, fr, 'steelblue', linewidth=0.8)
    axes[1].axhline(np.nanmean(fr), color='gray', linestyle='-', linewidth=0.5, alpha=0.5)
    axes[1].axhline(thr_low, color='red', linestyle='--', linewidth=0.8,
                     label=f'μ - {sigma}σ = {thr_low:.2f}')
    axes[1].axhline(thr_high, color='orange', linestyle='--', linewidth=0.8,
                     label=f'μ + {sigma}σ = {thr_high:.2f}')
    
    anom_low = anomalies[anomalies['type'] == 'low']
    anom_high = anomalies[anomalies['type'] == 'high']
    
    if not anom_low.empty:
        anom_idx = [list(dates).index(d) for d in anom_low['date'] if d in dates]
        axes[1].scatter(dates[anom_idx], fr[anom_idx], c='red', s=35, zorder=5,
                       label=f'Low anomalies ({len(anom_low)})')
    
    if not anom_high.empty:
        anom_idx = [list(dates).index(d) for d in anom_high['date'] if d in dates]
        axes[1].scatter(dates[anom_idx], fr[anom_idx], c='orange', s=35, zorder=5,
                       label=f'High anomalies ({len(anom_high)})')
    
    axes[1].set_ylabel('Forman-Ricci\nCurvature')
    axes[1].set_title('(b) Local Forman-Ricci with Anomaly Detection', loc='left')
    axes[1].legend(loc='upper right', fontsize=8)
    axes[1].grid(True, alpha=0.3)
    
    # Panel c: Dst
    if 'Dst' in omni_df.columns:
        dst_at_dates = omni_df['Dst'].reindex(dates)
        axes[2].plot(dates, dst_at_dates, 'darkgreen', linewidth=0.8)
        axes[2].axhline(0, color='gray', linestyle=':', linewidth=0.5)
        axes[2].set_ylabel('Dst Index (nT)')
        axes[2].set_title('(c) Dst Index (Geomagnetic Storm Indicator)', loc='left')
        axes[2].grid(True, alpha=0.3)
    
    axes[2].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.xticks(rotation=45)
    
    plt.tight_layout()
    plt.savefig(f'{save_prefix}_anomaly_detection_{station_label}.png')
    plt.close()
    print(f"  Saved: {save_prefix}_anomaly_detection_{station_label}.png")
    
    return anomalies


def plot_ricci_flow_detail(result, station_label, save_prefix='fig'):
    """
    Plot Ricci Flow local values if available.
    """
    if result['RF_node'] is None:
        print(f"  RF not computed for {station_label}, skipping plot.")
        return
    
    fig, axes = plt.subplots(2, 1, figsize=(14, 8))
    fig.suptitle(f'Ricci Flow Analysis ({station_label})', fontsize=13, fontweight='bold')
    
    dates = result['dates']
    
    axes[0].plot(dates, result['values'], 'b-o', markersize=3, linewidth=0.8)
    axes[0].set_ylabel('CR Counts')
    axes[0].set_title('(a) CR Counts', loc='left')
    axes[0].grid(True, alpha=0.3)
    
    rf = result['RF_node']
    colors = ['coral' if v > np.nanmean(rf) else 'steelblue' for v in rf]
    axes[1].bar(dates, rf, width=5, color=colors, alpha=0.7, edgecolor='none')
    axes[1].axhline(np.nanmean(rf), color='gray', linestyle='--', linewidth=0.8,
                     label=f'Mean = {np.nanmean(rf):.3f}')
    axes[1].set_ylabel('Ricci Flow\n(mean edge weight)')
    axes[1].set_title('(b) Local Ricci Flow per Node', loc='left')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    
    plt.tight_layout()
    plt.savefig(f'{save_prefix}_ricci_flow_{station_label}.png')
    plt.close()
    print(f"  Saved: {save_prefix}_ricci_flow_{station_label}.png")


def plot_curvature_distribution(results_dict, station_labels, save_prefix='fig'):
    """
    Plot histograms of curvature distributions per station.
    """
    n_stations = len(station_labels)
    fig, axes = plt.subplots(1, n_stations, figsize=(6 * n_stations, 5))
    if n_stations == 1:
        axes = [axes]
    
    fig.suptitle('Distribution of Local Forman-Ricci Curvature', fontsize=13, fontweight='bold')
    
    for i, label in enumerate(station_labels):
        fr = results_dict[label]['FR_node']
        fr_clean = fr[~np.isnan(fr)]
        
        axes[i].hist(fr_clean, bins=30, color='steelblue', alpha=0.7, edgecolor='white', density=True)
        axes[i].axvline(np.mean(fr_clean), color='red', linestyle='--', linewidth=1.5,
                        label=f'μ = {np.mean(fr_clean):.2f}')
        axes[i].axvline(np.median(fr_clean), color='orange', linestyle='--', linewidth=1.5,
                        label=f'median = {np.median(fr_clean):.2f}')
        axes[i].set_xlabel('Forman-Ricci Curvature')
        axes[i].set_ylabel('Density')
        axes[i].set_title(f'{label} (N={len(fr_clean)})')
        axes[i].legend(fontsize=9)
        axes[i].grid(True, alpha=0.2)
    
    plt.tight_layout()
    plt.savefig(f'{save_prefix}_curvature_distribution.png')
    plt.close()
    print(f"  Saved: {save_prefix}_curvature_distribution.png")


# ============================================================
# SECTION 7: MAIN EXECUTION
# ============================================================

def run_full_analysis(filepath, cr_stations=['JUNG', 'MOSC'],
                      cycles=None, freq='W',
                      compute_or=True, compute_rf=False,
                      alpha=0.5, rf_iterations=20, rf_max_nodes=80,
                      or_max_edges=600, sigma_anomaly=2.0,
                      output_dir='./output'):
    """
    Run the complete local curvature analysis pipeline.
    
    Parameters
    ----------
    filepath : str
        Path to merged NMDB + OMNI data file
    cr_stations : list of str
        Station column names (e.g., ['JUNG', 'MOSC'])
    cycles : list of int or None
        Solar cycles to analyze. None = all available.
    freq : str
        Resampling frequency: 'D' (daily), 'W' (weekly), 'M' (monthly)
    compute_or : bool
        Compute Ollivier-Ricci (slower)
    compute_rf : bool
        Compute Ricci Flow (slowest, only for small series)
    alpha : float
        Laziness parameter for OR and RF
    rf_iterations : int
    rf_max_nodes : int
    or_max_edges : int
    sigma_anomaly : float
        Sigma threshold for anomaly detection
    output_dir : str
    
    Returns
    -------
    all_results : dict
        Nested dict: {cycle: {station: result_dict}}
    all_correlations : pd.DataFrame
    all_anomalies : pd.DataFrame
    """
    os.makedirs(output_dir, exist_ok=True)
    save_prefix = os.path.join(output_dir, 'fig')
    
    print("=" * 70)
    print("LOCAL CURVATURE ANALYSIS OF COSMIC RAY TIME SERIES")
    print("=" * 70)
    
    # --- Load data ---
    print("\n[1] Loading data...")
    df = load_and_prepare_data(filepath)
    
    # --- Determine cycles ---
    if cycles is None:
        cycles = sorted(df['Cycle'].dropna().unique().astype(int))
    print(f"\nCycles to analyze: {cycles}")
    print(f"Resampling frequency: {freq}")
    print(f"Stations: {cr_stations}")
    
    # --- Resample ---
    print(f"\n[2] Resampling to {freq}...")
    df_resampled = resample_data(df, freq=freq)
    print(f"  Resampled shape: {df_resampled.shape}")
    
    # --- Process each cycle and station ---
    all_results = {}
    all_correlations = []
    all_anomalies = []
    
    for cycle in cycles:
        print(f"\n{'='*50}")
        print(f"  CYCLE {cycle}")
        print(f"{'='*50}")
        
        cycle_results = {}
        
        for station in cr_stations:
            if station not in df_resampled.columns:
                print(f"  WARNING: {station} not in data, skipping.")
                continue
            
            print(f"\n  --- {station} (Cycle {cycle}) ---")
            
            series, dates = get_cycle_data(df_resampled, cycle, cr_col=station)
            
            if series is None:
                continue
            
            print(f"  Series length: {len(series)} ({dates[0].strftime('%Y-%m')} to {dates[-1].strftime('%Y-%m')})")
            
            # Build curvatures
            result = build_curvature_timeseries(
                series.values, dates,
                compute_or=compute_or,
                compute_rf=compute_rf,
                alpha=alpha,
                rf_iterations=rf_iterations,
                rf_max_nodes=rf_max_nodes,
                or_max_edges=or_max_edges,
                verbose=True
            )
            
            cycle_results[station] = result
            
            # Correlations with OMNI
            print(f"\n  Computing correlations with OMNI variables...")
            corr_fr = compute_correlations(result['FR_node'], df_resampled, dates, curv_label='FR')
            corr_fr['Station'] = station
            corr_fr['Cycle'] = cycle
            all_correlations.append(corr_fr)
            
            if result['OR_node'] is not None:
                corr_or = compute_correlations(result['OR_node'], df_resampled, dates, curv_label='OR')
                corr_or['Station'] = station
                corr_or['Cycle'] = cycle
                all_correlations.append(corr_or)
            
            # Anomaly detection
            anomalies, _, _ = detect_curvature_anomalies(
                result['FR_node'], dates, sigma=sigma_anomaly, metric_name='FR'
            )
            anomalies['Station'] = station
            anomalies['Cycle'] = cycle
            all_anomalies.append(anomalies)
            
            print(f"  Anomalies detected: {len(anomalies)} "
                  f"({len(anomalies[anomalies['type']=='low'])} low, "
                  f"{len(anomalies[anomalies['type']=='high'])} high)")
        
        all_results[cycle] = cycle_results
        
        # --- Generate plots per cycle ---
        if cycle_results:
            print(f"\n  Generating plots for cycle {cycle}...")
            
            cycle_prefix = os.path.join(output_dir, f'cycle{cycle}')
            
            # Curvature time series
            plot_curvature_timeseries(cycle_results, list(cycle_results.keys()),
                                      save_prefix=cycle_prefix)
            
            # Curvature vs OMNI
            for station in cycle_results:
                plot_curvature_vs_omni(cycle_results[station], df_resampled,
                                       station, save_prefix=cycle_prefix)
                
                plot_anomaly_detection(cycle_results[station], df_resampled,
                                       station, sigma=sigma_anomaly,
                                       save_prefix=cycle_prefix)
                
                if cycle_results[station]['RF_node'] is not None:
                    plot_ricci_flow_detail(cycle_results[station], station,
                                           save_prefix=cycle_prefix)
            
            # Distribution
            plot_curvature_distribution(cycle_results, list(cycle_results.keys()),
                                        save_prefix=cycle_prefix)
            
            # Cross-station correlation
            stations_in_cycle = list(cycle_results.keys())
            if len(stations_in_cycle) >= 2:
                cross_corr = compute_cross_station_correlation(
                    cycle_results[stations_in_cycle[0]],
                    cycle_results[stations_in_cycle[1]],
                    stations_in_cycle[0], stations_in_cycle[1]
                )
                if not cross_corr.empty:
                    cross_corr['Cycle'] = cycle
                    all_correlations.append(cross_corr)
    
    # --- Compile results ---
    print("\n" + "=" * 70)
    print("SUMMARY")
    print("=" * 70)
    
    if all_correlations:
        df_corr = pd.concat(all_correlations, ignore_index=True)
        df_corr.to_csv(os.path.join(output_dir, 'correlations.csv'), index=False)
        print("\nCorrelation Results:")
        print(df_corr.to_string(index=False))
    else:
        df_corr = pd.DataFrame()
    
    if all_anomalies:
        df_anom = pd.concat(all_anomalies, ignore_index=True)
        df_anom.to_csv(os.path.join(output_dir, 'anomalies.csv'), index=False)
        print(f"\nTotal anomalies detected: {len(df_anom)}")
    else:
        df_anom = pd.DataFrame()
    
    print(f"\nAll outputs saved to: {output_dir}/")
    print("=" * 70)
    
    return all_results, df_corr, df_anom

In [2]:
# ============================================================
# USAGE EXAMPLE
# ============================================================

if __name__ == "__main__":
    """
    USAGE:
    ------
    1. Save your merged DataFrame as CSV:
       
       df.to_csv('nmdb_omni_data.csv')
    
    2. Run this script:
    
       python curvature_analysis_real.py
    
    3. Or import and call directly:
    
       from curvature_analysis_real import run_full_analysis
       results, correlations, anomalies = run_full_analysis(
           'nmdb_omni_data.csv',
           cr_stations=['JUNG', 'MOSC'],
           cycles=[24, 25],
           freq='W',
           compute_or=True,
           compute_rf=False,
           output_dir='./output_curvature'
       )
    
    CONFIGURATION TIPS:
    -------------------
    - freq='W' (weekly) is recommended for a first pass.
      Weekly gives ~500 nodes per cycle, which is manageable.
    
    - freq='D' (daily) gives ~4000 nodes per cycle.
      Forman-Ricci is still fast, but Ollivier-Ricci will be slow.
      Set or_max_edges=1000 or compute_or=False for daily.
    
    - Ricci Flow is very expensive. Only enable for small subsets
      (compute_rf=True, rf_max_nodes=100).
    
    - For a quick exploration: freq='M', compute_or=False, compute_rf=False
      This only computes Forman-Ricci on monthly visibility graphs.
    
    SUGGESTED ANALYSIS PLAN:
    -------------------------
    Phase 1: Quick overview
        run_full_analysis(..., freq='M', compute_or=False, cycles=[24])
    
    Phase 2: Weekly resolution with both curvatures  
        run_full_analysis(..., freq='W', compute_or=True, cycles=[24, 25])
    
    Phase 3: Daily resolution, Forman-Ricci only
        run_full_analysis(..., freq='D', compute_or=False, cycles=[24])
    
    Phase 4: Ricci Flow on specific interesting periods
        # Extract a 6-month window around a major storm, then:
        build_curvature_timeseries(subset, dates, compute_rf=True, rf_max_nodes=200)
    """
    
    # ---- MODIFY THIS PATH TO YOUR DATA ----
    DATA_PATH = 'All_Data.csv'
    
    # ---- CONFIGURATION ----
    CONFIG = {
        'cr_stations': ['JUNG', 'MOSC'],
        'cycles': [24, 25],          # or None for all
        'freq': 'M',                  # 'D', 'W', or 'M'
        'compute_or': True,           # Ollivier-Ricci (slower)
        'compute_rf': False,          # Ricci Flow (very slow, small series only)
        'alpha': 0.5,                 # laziness parameter
        'rf_iterations': 20,
        'rf_max_nodes': 80,
        'or_max_edges': 600,
        'sigma_anomaly': 2.0,
        'output_dir': './output_curvature_Montly'
    }
    
    results, correlations, anomalies = run_full_analysis(DATA_PATH, **CONFIG)

LOCAL CURVATURE ANALYSIS OF COSMIC RAY TIME SERIES

[1] Loading data...
Data loaded: 22646 rows, 9 columns
Date range: 1964-01-01 00:00:00 to 2025-12-31 00:00:00
Cycles present: [np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25)]

Missing data (%):
  JUNG                          : 0.3%
  MOSC                          : 1.6%
  B                             : 11.6%
  Temp                          : 17.0%
  Density                       : 14.7%
  Speed                         : 11.5%
  SSN                           : 0.0%
  Dst                           : 0.0%
  Cycle                         : 0.0%

Cycles to analyze: [24, 25]
Resampling frequency: M
Stations: ['JUNG', 'MOSC']

[2] Resampling to M...
  Resampled shape: (744, 9)

  CYCLE 24

  --- JUNG (Cycle 24) ---
  Series length: 132 (2008-01 to 2018-12)
  Building VG (132 nodes)...
  VG: 132 nodes, 655 edges (0.0s)
  FR: mean=-6.623, std=5.754 (0.0s)
  [OR] Computing shortest paths (132 nodes)...
  [O